# R4-N3: fresh Colab install and production verification

Run this notebook in a newly created Colab runtime. It installs the pinned app environment, retrieves the production Protocol B bundle from the shared Drive project, runs the CLI and Streamlit AppTest verification, and writes command output under `reports/r4_n3/`. A successful output is evidence only for the recorded runtime and lock.

Sync this repository to the Drive project folder before the run. It must contain `requirements/app.lock.txt` and the production bundle under `bundles/r3_n1_wavlm_head_B_calibrated/`. The expected fixture record must already have been populated from a separate verified CLI run. This notebook does not invent or self-record expected model predictions.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
assert (PROJECT_ROOT / 'REVISED_PLAN.md').is_file(), f'Repository not found at {PROJECT_ROOT}'
print('Project:', PROJECT_ROOT)
print('Python:', __import__('sys').version)

## Install in this runtime

The supported pinned runtime is Python 3.13, matching the recorded Colab reproduction baseline. The setup script creates a virtual environment; the following cells run all reproduction commands through that environment.

In [ ]:
import subprocess, sys
VENV = Path('/content/echoproof-r4-n3-venv')
setup = subprocess.run(['bash', 'scripts/setup_env.sh', str(VENV)], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(setup.stdout)
print(setup.stderr)
if setup.returncode: raise RuntimeError(f'setup_env.sh failed with exit {setup.returncode}')
VENV_PY = VENV / 'bin/python'
print(subprocess.check_output([str(VENV_PY), '--version'], text=True))

## Retrieve the locked production bundle

The source folder is local to the mounted project Drive. Retrieval verifies the manifest and every component against `locks/B_lock_gr3.json`; no model download occurs.

In [ ]:
bundle_source = PROJECT_ROOT / 'bundles/r3_n1_wavlm_head_B_calibrated'
fetch = subprocess.run([str(VENV_PY), 'scripts/fetch_bundles.py', '--source', str(bundle_source)], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(fetch.stdout)
print(fetch.stderr)
if fetch.returncode: raise RuntimeError(f'bundle retrieval failed with exit {fetch.returncode}')

## CLI and app verification

The verifier checks the locked bundle, stored fixture score and decision, and Streamlit AppTest. Its JSON output is saved even when verification fails, so the evidence distinguishes a pass from an incomplete run.

In [ ]:
import datetime, json, platform
evidence_dir = PROJECT_ROOT / 'reports/r4_n3'
evidence_dir.mkdir(parents=True, exist_ok=True)
evidence_file = evidence_dir / ('colab_fresh_install_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.json')
result = subprocess.run([str(VENV_PY), 'scripts/verify_install.py'], cwd=PROJECT_ROOT, text=True, capture_output=True)
record = {'status': 'passed' if result.returncode == 0 else 'failed', 'exit_code': result.returncode, 'python': platform.python_version(), 'platform': platform.platform(), 'stdout': result.stdout, 'stderr': result.stderr}
evidence_file.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
print(json.dumps(record, indent=2))
print('Evidence saved:', evidence_file)
if result.returncode: raise RuntimeError(f'verification failed with exit {result.returncode}')

## Evidence handling

Save this executed notebook and the JSON report path printed above. Review the status and stderr; do not report a failed, skipped, or unexecuted cell as fresh-environment verification. The Drive folder should contain the exact lock files used by this run.